In [35]:
import numpy as np
import h5py
import json
import os
import torch
from PIL import Image
import cv2

from orion.utils.misc_utils import (
    get_all_annotation_folder_info,
    plotly_draw_3d_pcd,
    outlier_detection,
    transform_points,
    visualize_plane_detection,
    objective_3d_to_2d_err,
    project_2d
)
from orion.utils.o3d_utils import (
    O3DPointCloud,
    estimate_rotation,
    get_3d_traj_from_2d_tracking,
)
import trimesh
import open3d as o3d
import re
import PIL
from PIL import Image

import os
from pathlib import Path
import cv2
import imageio
import math
from tqdm import tqdm
import point_cloud_utils as pcu
from hausdorff import hausdorff_distance
import plotly.graph_objects as go

from scipy.optimize import minimize

In [36]:
annotation_folder = "/home/arisrei/research/RPL/ORION/annotations/human_demo/long_horizon_rearrangement_cheese_4_demo_mmde"
images, depths, masks, K, tracked_points = get_all_annotation_folder_info(annotation_folder, force_mmde=False)
tracked_points = tracked_points["pred_tracks"][0]
start_idx = 0
end_idx = -1
obj_idx = 0
obj_tracked_points = tracked_points[:, obj_idx*40:(obj_idx+1)*40,:]

In [ ]:
display(Image.fromarray(images[start_idx]))

In [ ]:

img = images[end_idx].copy()
frame_tracked_points = obj_tracked_points[end_idx]

# draw circles on the image for the tracked points
for point in frame_tracked_points:
    cv2.circle(img, (int(point[0]), int(point[1])), 2, (0, 255, 0), -1)

outlier_mask = outlier_detection(frame_tracked_points, threshold=1)

# draw circles on the image for the outlier tracked points
for point in frame_tracked_points[outlier_mask]:
    cv2.circle(img, (int(point[0]), int(point[1])), 2, (255, 0, 0), -1)

display(Image.fromarray(img))


In [ ]:
start_2d_pts = obj_tracked_points[start_idx][np.logical_not(outlier_mask)]
end_2d_pts = obj_tracked_points[end_idx][np.logical_not(outlier_mask)]

trajs_3d = get_3d_traj_from_2d_tracking(np.array([start_2d_pts, end_2d_pts]), depths[[start_idx, end_idx]], K)
print(trajs_3d.shape)

#all_poses = np.load("/home/arisrei/research/RPL/ORION/annotations/human_demo/long_horizon_rearrangement_cheese_4_demo/FoundationPose/USING_FOR_SIM_tracking_long_horizon_rearrangement_cheese_4_demo_no_depth/all_poses.npy")



In [ ]:
frame_pcd = O3DPointCloud(max_points=10000)
frame_pcd.create_from_rgbd(
    color=images[start_idx], depth=depths[start_idx], intrinsic_matrix=K, depth_trunc=1.5
)
plane_estimation_kwargs = {
    "ransac_n": 3,
    "num_iterations": 1000,
    "distance_threshold": 0.005,
}
plane_estimation_result = frame_pcd.plane_estimation(**plane_estimation_kwargs)
T_xy_plane_align = estimate_rotation(plane_estimation_result["plane_model"], z_up=False)
frame_pcd.preprocess()

num_traj_frames = trajs_3d.shape[0]
trajs_3d = trajs_3d.reshape(-1, 3)
#trajs_3d = transform_points(T_xy_plane_align, trajs_3d)
trajs_3d = trajs_3d.reshape(num_traj_frames, -1, 3)
#frame_pcd.transform(T_xy_plane_align)

fig = plotly_draw_3d_pcd(
    frame_pcd.get_points(),
    frame_pcd.get_colors(),
    offline=True,
    addition_points=np.transpose(trajs_3d, (1, 0, 2)),
    additional_point_draw_lines=True,
)

plane_surface = visualize_plane_detection(plane_estimation_result)
#fig.add_trace(plane_surface)
display(fig.show())

In [ ]:
#avg_start_end = np.mean(trajs_3d, axis=1)
trajectory = np.linspace(trajs_3d[0][0], trajs_3d[1][0], num=images.shape[0])

# normalize the 2d trajectory


img = images[start_idx].copy()
for i, point in enumerate(trajectory):
    projected_2d = project_2d(point[0], point[1], point[2], K[0, 0], K[1, 1], K[0, 2], K[1, 2])
    cv2.circle(img, (int(projected_2d[0]), int(projected_2d[1])), 2, (0, 255, 0), -1)
    
    tracked_2d = obj_tracked_points[i][np.logical_not(outlier_mask)][0]
    cv2.circle(img, (int(tracked_2d[0]), int(tracked_2d[1])), 2, (255, 0, 0), -1)

display(Image.fromarray(img))


In [ ]:
keypoints = []
for i in range(obj_tracked_points.shape[0]):
    keypoints.append(obj_tracked_points[i][np.logical_not(outlier_mask)][0])


shortest_path_weight = .5
result = minimize(
    objective_3d_to_2d_err,
    trajectory.reshape(-1),
    args=(keypoints, K, shortest_path_weight), #2d keypoints, intrinsic matrix, coefficient for shortest path heuristic
    options={"disp": True, "maxiter": 10000},
    method="L-BFGS-B"
)
result.x = result.x.reshape(-1, 3)

In [ ]:
#avg_start_end = np.mean(trajs_3d, axis=1)
trajectory = np.linspace(trajs_3d[0][0], trajs_3d[1][0], num=images.shape[0])

img = images[start_idx].copy()
for i, point in enumerate(trajectory):
    # original linear interpolation
    projected_2d = project_2d(point[0], point[1], point[2], K[0, 0], K[1, 1], K[0, 2], K[1, 2])
    cv2.circle(img, (int(projected_2d[0]), int(projected_2d[1])), 2, (0, 255, 0), -1)
    
    # tracked 2d trajectory
    tracked_2d = obj_tracked_points[i][np.logical_not(outlier_mask)][0]
    cv2.circle(img, (int(tracked_2d[0]), int(tracked_2d[1])), 2, (255, 0, 0), -1)
    
    # optimized trajectory backprojected to 2d
    optimized_2d = project_2d(result.x[i][0], result.x[i][1], result.x[i][2], K[0, 0], K[1, 1], K[0, 2], K[1, 2])
    cv2.circle(img, (int(optimized_2d[0]), int(optimized_2d[1])), 2, (0, 0, 255), -1)
    

display(Image.fromarray(img))


In [ ]:
fig = plotly_draw_3d_pcd(
    frame_pcd.get_points(),
    frame_pcd.get_colors(),
    offline=True,
    addition_points=result.x.reshape(1, -1, 3),
    additional_point_draw_lines=True,
)
display(fig.show())


In [45]:
fig.write_html(f"junk/optimized_traj_shortest_path_weight_{shortest_path_weight}.html")